# **Welcome to Nahj**

Note:

1. Please check that you are running on T4 GPU. (Runtime > Change runtime type >T4 GPU > Save.)

2. Run the last cell below and click on the gradio link that will appear

(For a better experience, copy gradio link and copy it in a new webpage.)



In [ ]:

# ==============================================================================
# CELL 0.1 — Mount Google Drive
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')

import os

# All outputs go to Drive — nothing is saved locally
DRIVE_BASE = "/content/drive/MyDrive/allam_mcq_project"
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/data", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/adapter", exist_ok=True)
os.makedirs(f"{DRIVE_BASE}/rag", exist_ok=True)

print(f"Project folder ready: {DRIVE_BASE}")

In [ ]:
# CELL 0.2 — Load Dataset from Google Sheets
# ==============================================================================

import pandas as pd

url = "https://docs.google.com/spreadsheets/d/18SE0XvxEpYx-8ZDg76JTpQ-TUjGem1I0uAyMGE9qWQ4/export?format=csv"
df = pd.read_csv(url)

# Standardize column names immediately
df.columns = df.columns.str.strip().str.lower()

print(f"Raw dataset shape : {df.shape}")
print(f"Columns           : {list(df.columns)}")
print(f"\nFirst 2 rows:")
df.head(2)


# ==============================================================================

In [ ]:
# CELL 1.1 — Data Cleaning (FIXED: dedup + text normalization + quality report)
# ==============================================================================

import pandas as pd
import re

# ── 1. Drop rows missing critical fields ──────────────────────────────────────
before = len(df)
df = df.dropna(subset=['slide_text', 'question', 'answer'])
print(f"Dropped (null critical fields) : {before - len(df)} rows")

# ── 2. Normalize text fields — strip whitespace, collapse internal spaces ──────
def clean_text(text: str) -> str:
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)        # collapse multiple spaces/newlines
    text = re.sub(r'<[^>]+>', '', text)     # strip any HTML tags
    return text

df['slide_text'] = df['slide_text'].apply(clean_text)
df['question']   = df['question'].apply(clean_text)
df['answer']     = df['answer'].apply(clean_text)

# ── 3. Drop rows where answer is a placeholder ─────────────────────────────────
bad_answers = ['nan', 'none', 'n/a', '', '-']
before = len(df)
df = df[~df['answer'].str.lower().isin(bad_answers)]
print(f"Dropped (bad answers)          : {before - len(df)} rows")

# ── 4. Drop exact duplicate (slide_text + question) pairs ─────────────────────
before = len(df)
df = df.drop_duplicates(subset=['slide_text', 'question'])
print(f"Dropped (duplicates)           : {before - len(df)} rows")

# ── 5. Drop rows with suspiciously short questions (<10 chars) ─────────────────
before = len(df)
df = df[df['question'].str.len() >= 10]
print(f"Dropped (short questions)      : {before - len(df)} rows")

# ── 6. Reset index ────────────────────────────────────────────────────────────
df = df.reset_index(drop=True)

# ── 7. Quality report ─────────────────────────────────────────────────────────
print(f"\n{'='*40}")
print(f"Final clean dataset            : {len(df)} rows")
print(f"Unique CLO IDs                 : {df['clo_id'].nunique()}")
print(f"Avg slide_text length (chars)  : {df['slide_text'].str.len().mean():.0f}")
print(f"Avg question length (chars)    : {df['question'].str.len().mean():.0f}")
print(f"Unique answers                 : {df['answer'].nunique()}")
print(f"{'='*40}")

# Save clean df to Drive
df.to_csv(f"{DRIVE_BASE}/data/clean_dataset.csv", index=False)
print(f"Saved: {DRIVE_BASE}/data/clean_dataset.csv")


In [ ]:
# CELL 1.2 — EDA (lightweight, no external libs beyond matplotlib)
# ==============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("Dataset EDA", fontsize=14, fontweight='bold')

# Slide text length distribution
df['slide_text'].str.len().hist(ax=axes[0,0], bins=30, color='#4F8EF7', edgecolor='white')
axes[0,0].set_title('Slide Text Length (chars)')
axes[0,0].set_xlabel('Characters')

# Question length distribution
df['question'].str.len().hist(ax=axes[0,1], bins=30, color='#00C9B1', edgecolor='white')
axes[0,1].set_title('Question Length (chars)')
axes[0,1].set_xlabel('Characters')

# CLO distribution
df['clo_id'].value_counts().sort_index().plot(kind='bar', ax=axes[1,0], color='#F59E0B')
axes[1,0].set_title('Samples per CLO ID')
axes[1,0].set_xlabel('CLO ID')
axes[1,0].tick_params(axis='x', rotation=45)

# Answer length distribution
df['answer'].str.len().hist(ax=axes[1,1], bins=30, color='#EF4444', edgecolor='white')
axes[1,1].set_title('Answer Length (chars)')
axes[1,1].set_xlabel('Characters')

plt.tight_layout()
plt.savefig(f"{DRIVE_BASE}/data/eda.png", dpi=120, bbox_inches='tight')
plt.show()
print(f"Saved EDA plot: {DRIVE_BASE}/data/eda.png")

# Print CLO breakdown
print("\nSamples per CLO ID:")
print(df['clo_id'].value_counts().sort_index().to_string())



In [ ]:
# CELL 1.3 — Install libraries needed for preprocessing onward
# ==============================================================================

!pip install -q transformers==4.44.2
!pip install -q sentencepiece protobuf
!pip install -q langchain langchain-core
!pip install -q datasets==2.21.0

print("Libraries installed.")



In [ ]:
# ==============================================================================
# CELL 1.3b — Data Preprocessing
# ==============================================================================

import pandas as pd
import json
from transformers import AutoTokenizer

# ── Load ALLaM tokenizer — the ONLY correct tokenizer to use ──────────────────
MODEL_ID = "humain-ai/ALLaM-7B-Instruct-preview"
print("Loading ALLaM tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

# ── Token counter using ALLaM tokenizer (not tiktoken/gpt) ────────────────────
def count_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))

# ── Load clean dataset from 1.1 ───────────────────────────────────────────────
df = pd.read_csv(f"{DRIVE_BASE}/data/clean_dataset.csv")
df.columns = df.columns.str.strip().str.lower()
print(f"Loaded {len(df)} clean rows")

# ── Subject mapping — explicit lookup, not a fragile threshold ─────────────────
# Map each clo_id to its subject based on your actual dataset structure.
# Edit this dictionary if your CLO IDs differ.
CLO_SUBJECT_MAP = {clo_id: "Data Structures" for clo_id in range(1, 18)}
CLO_SUBJECT_MAP.update({clo_id: "Compiler Construction" for clo_id in range(18, 40)})

def get_subject(clo_id) -> str:
    try:
        return CLO_SUBJECT_MAP.get(int(float(clo_id)), "Unknown")
    except (ValueError, TypeError):
        return "Unknown"

# ── Build structured RAG chunks ───────────────────────────────────────────────
final_chunks = []
token_counts = []

for _, row in df.iterrows():
    slide_text = str(row['slide_text'])
    question   = str(row['question'])
    answer     = str(row['answer'])
    subject    = get_subject(row['clo_id'])

    page_content = f"Slide Content: {slide_text}\nQuestion: {question}\nAnswer: {answer}"
    tokens       = count_tokens(page_content)
    token_counts.append(tokens)

    chunk_data = {
        "chunk_content": page_content,
        "metadata": {
            "subject":      subject,
            "clo":          str(row['clo']) if pd.notna(row['clo']) else "",
            "clo_id":       int(float(row['clo_id'])) if pd.notna(row['clo_id']) else 0,
            "slide_index":  str(row['source_slides']) if pd.notna(row['source_slides']) else "",
            "tokens_count": tokens,
        }
    }
    final_chunks.append(chunk_data)

# ── Token length report ───────────────────────────────────────────────────────
import numpy as np
token_counts = np.array(token_counts)
print(f"\nToken count stats (ALLaM tokenizer):")
print(f"  Min    : {token_counts.min()}")
print(f"  Max    : {token_counts.max()}")
print(f"  Mean   : {token_counts.mean():.1f}")
print(f"  >512   : {(token_counts > 512).sum()} chunks ({(token_counts > 512).mean()*100:.1f}%)")
print(f"  >1024  : {(token_counts > 1024).sum()} chunks — these will be truncated during training")

# ── Subject distribution report ───────────────────────────────────────────────
subjects = [c['metadata']['subject'] for c in final_chunks]
from collections import Counter
print(f"\nSubject distribution:")
for subj, count in Counter(subjects).items():
    print(f"  {subj}: {count} chunks")
print(f"  Unknown: {subjects.count('Unknown')} — fix CLO_SUBJECT_MAP if this is > 0")

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = f"{DRIVE_BASE}/data/structured_rag_chunks.json"
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(final_chunks, f, ensure_ascii=False, indent=2)

print(f"\nSaved {len(final_chunks)} chunks → {out_path}")


In [ ]:
# CELL 1.4 — Data Augmentation
# Approach: use the base ALLaM model to generate question paraphrases.
# ==============================================================================

# NOTE: Run this cell AFTER 2.1 (model loading) if you want to use ALLaM itself.


import json
import copy
import random
from collections import defaultdict

random.seed(42)

with open(f"{DRIVE_BASE}/data/structured_rag_chunks.json", "r", encoding="utf-8") as f:
    structured_rag_chunks = json.load(f)

print(f"Original chunks: {len(structured_rag_chunks)}")

# ── Helper: parse chunk into parts ────────────────────────────────────────────
def parse_chunk(chunk_content: str):
    import re
    m = re.match(r'Slide Content:\s*(.*?)\nQuestion:\s*(.*?)\nAnswer:\s*(.*)', chunk_content, re.DOTALL)
    if m:
        return m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
    return chunk_content, '', ''

def rebuild_chunk(slide: str, question: str, answer: str) -> str:
    return f"Slide Content: {slide}\nQuestion: {question}\nAnswer: {answer}"

def make_aug_item(original: dict, new_content: str, aug_type: str) -> dict:
    new_item = copy.deepcopy(original)
    new_item["chunk_content"] = new_content
    new_item["metadata"]["augmentation_type"] = aug_type
    return new_item

# ── Augmentation 1: Question rephrasing templates ─────────────────────────────
# Transform "What is X?" → "Define X." → "Which of the following describes X?"
# This creates genuinely different training examples for the same content.

def rephrase_question(question: str) -> list[str]:
    """Generate up to 2 rephrased versions of a question."""
    q = question.strip().rstrip('?').rstrip('.')
    variants = []

    # Template A: "Define ..." → "What is meant by ...?"
    if q.lower().startswith("what is"):
        topic = q[7:].strip()
        variants.append(f"Define {topic}.")
        variants.append(f"Which of the following best describes {topic}?")

    # Template B: "Define ..." → "What is ...?"
    elif q.lower().startswith("define"):
        topic = q[6:].strip()
        variants.append(f"What is {topic}?")
        variants.append(f"Which of the following best describes {topic}?")

    # Template C: "Explain ..." → "What does ... mean?"
    elif q.lower().startswith("explain"):
        topic = q[7:].strip()
        variants.append(f"What does {topic} mean?")

    # Template D: Generic → add "In the context of [subject],"
    if len(variants) < 2:
        variants.append(f"In the context of computer science, {question.lower()}")

    return variants[:2]  # max 2 paraphrases per original

# ── Augmentation 2: Cross-CLO negative sampling ───────────────────────────────
# Group chunks by CLO — use answers from DIFFERENT CLOs as distractors.
# This gives you distractor material for free, no API needed.

clo_to_answers = defaultdict(list)
for chunk in structured_rag_chunks:
    _, _, answer = parse_chunk(chunk["chunk_content"])
    clo_id = chunk["metadata"]["clo_id"]
    if answer and answer.lower() not in ['nan', 'none', '']:
        clo_to_answers[clo_id].append(answer)

def get_distractors(clo_id: int, correct_answer: str, n: int = 3) -> list[str]:
    """Get n wrong answers from OTHER CLOs."""
    other_answers = []
    for other_clo, answers in clo_to_answers.items():
        if other_clo != clo_id:
            other_answers.extend(answers)
    # Remove the correct answer just in case
    other_answers = [a for a in other_answers if a.lower() != correct_answer.lower()]
    if len(other_answers) < n:
        return other_answers
    return random.sample(other_answers, n)

# ── Apply augmentation ────────────────────────────────────────────────────────
augmented_data = []
counts = {'original': 0, 'question_rephrase': 0}

for item in structured_rag_chunks:
    slide, question, answer = parse_chunk(item["chunk_content"])

    # Always keep original
    orig = copy.deepcopy(item)
    orig["metadata"]["augmentation_type"] = "original"
    # Add distractors to metadata for use in Gradio MCQ formatting later
    orig["metadata"]["distractors"] = get_distractors(
        item["metadata"]["clo_id"], answer
    )
    augmented_data.append(orig)
    counts['original'] += 1

    # Add rephrased question variants (same slide, same answer, different Q)
    for rephrased_q in rephrase_question(question):
        if rephrased_q and rephrased_q != question:
            new_content = rebuild_chunk(slide, rephrased_q, answer)
            aug_item = make_aug_item(item, new_content, "question_rephrase")
            aug_item["metadata"]["distractors"] = get_distractors(
                item["metadata"]["clo_id"], answer
            )
            augmented_data.append(aug_item)
            counts['question_rephrase'] += 1

# ── Report ─────────────────────────────────────────────────────────────────────
print(f"\nOriginals          : {counts['original']}")
print(f"Question rephrases : {counts['question_rephrase']}")
print(f"Total              : {len(augmented_data)}")
print(f"Augmentation factor: {len(augmented_data) / counts['original']:.2f}x")

# ── Save ──────────────────────────────────────────────────────────────────────
out_path = f"{DRIVE_BASE}/data/augmented_rag_chunks.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(augmented_data, f, indent=2, ensure_ascii=False)

print(f"\nSaved → {out_path}")



In [ ]:
# ==============================================================================
# CELL 1.5 — Data Preparation: Convert to ALLaM ChatML JSONL
# ==============================================================================

import json
import random
import re

DRIVE_BASE = "/content/drive/MyDrive/allam_mcq_project"

with open(f"{DRIVE_BASE}/data/augmented_rag_chunks.json", "r", encoding="utf-8") as f:
    augmented_data = json.load(f)

print(f"Loaded {len(augmented_data)} augmented chunks")

# ── Dynamic system prompt builder ─────────────────────────────────────────────
def build_system_prompt(subject: str = None, clo: str = None) -> str:
    base = (
        "أنت مساعد تعليمي. مهمتك توليد أسئلة اختيار من متعدد "
        "بناءً على المحتوى التعليمي المقدم فقط. "
        "You are an educational assistant. Your task is to generate "
        "multiple-choice questions based only on the provided content. "
        "Do not use any outside knowledge. "
        "Always answer based strictly on the context given."
    )
    if subject and subject.lower() not in ["unknown", "", "none"]:
        base += f" The subject is: {subject}."
    if clo and clo.lower() not in ["", "none", "nan"]:
        base += f" Focus on this learning outcome: {clo}."
    return base

# ── Format function — one sample at a time ────────────────────────────────────
def chunk_to_instruction(chunk: dict):
    content = chunk["chunk_content"]
    meta    = chunk["metadata"]
    subject = meta.get("subject", "")
    clo     = meta.get("clo", "")

    m = re.match(
        r'Slide Content:\s*(.*?)\nQuestion:\s*(.*?)\nAnswer:\s*(.*)',
        content, re.DOTALL
    )
    if not m:
        return None

    slide_text = m.group(1).strip()
    question   = m.group(2).strip()
    answer     = m.group(3).strip()

    if not question or not answer or answer.lower() in ['nan', 'none', '']:
        return None

    # System prompt is built dynamically per sample
    system_content = build_system_prompt(subject=subject, clo=clo)

    user_message = (
        f"Content:\n{slide_text}\n\n"
        f"Generate a multiple-choice question with options A, B, C, D "
        f"and indicate the correct answer."
    )

    # Assistant response includes question + answer in structured format
    assistant_message = (
        f"Question: {question}\n"
        f"A) {answer}\n"
        f"B) —\n"
        f"C) —\n"
        f"D) —\n"
        f"Correct: A"
    )

    return {
        "messages": [
            {"role": "system",    "content": system_content},
            {"role": "user",      "content": user_message},
            {"role": "assistant", "content": assistant_message},
        ],
        "metadata": {
            "subject":           subject,
            "clo":               clo,
            "clo_id":            meta.get("clo_id", 0),
            "augmentation_type": meta.get("augmentation_type", "original"),
        }
    }

# ── Convert all chunks ────────────────────────────────────────────────────────
all_samples, skipped = [], 0
for chunk in augmented_data:
    sample = chunk_to_instruction(chunk)
    if sample:
        all_samples.append(sample)
    else:
        skipped += 1

print(f"Valid samples : {len(all_samples)}")
print(f"Skipped       : {skipped}")

# ── Shuffle and split 90/10 ───────────────────────────────────────────────────
random.seed(42)
random.shuffle(all_samples)
split_idx     = int(len(all_samples) * 0.9)
train_samples = all_samples[:split_idx]
val_samples   = all_samples[split_idx:]

print(f"Train set     : {len(train_samples)}")
print(f"Validation set: {len(val_samples)}")

# ── Save JSONL to Drive ───────────────────────────────────────────────────────
def save_jsonl(data, path):
    with open(path, "w", encoding="utf-8") as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

TRAIN_PATH = f"{DRIVE_BASE}/data/train.jsonl"
VAL_PATH   = f"{DRIVE_BASE}/data/val.jsonl"

save_jsonl(train_samples, TRAIN_PATH)
save_jsonl(val_samples,   VAL_PATH)

# ── Validation: re-read and verify ────────────────────────────────────────────
with open(TRAIN_PATH) as f:
    train_lines = f.readlines()
with open(VAL_PATH) as f:
    val_lines = f.readlines()

assert len(train_lines) == len(train_samples), "Train file line count mismatch!"
assert len(val_lines)   == len(val_samples),   "Val file line count mismatch!"

sample = json.loads(train_lines[0])
assert "messages" in sample,          "Missing 'messages' key!"
assert len(sample["messages"]) == 3,  "Expected 3 messages (system, user, assistant)!"

print(f"\n✓ Validation passed")
print(f"✓ Train : {len(train_lines)} lines → {TRAIN_PATH}")
print(f"✓ Val   : {len(val_lines)} lines → {VAL_PATH}")
print(f"\n--- Sample ---")
print(f"System : {sample['messages'][0]['content'][:120]}...")
print(f"User   : {sample['messages'][1]['content'][:100]}...")
print(f"Answer : {sample['messages'][2]['content'][:100]}...")




In [ ]:
# ==============================================================================
# CELL 2.1 — Environment Setup & Dependency Installation
# ==============================================================================
!pip uninstall -q -y transformers peft trl bitsandbytes accelerate datasets

!pip install -q "transformers==4.41.2"
!pip install -q "accelerate==0.34.0"
!pip install -q "bitsandbytes"
!pip install -q "peft==0.11.0"
!pip install -q "trl==0.9.4"
!pip install -q "datasets==2.19.0"
!pip install -q "sentencepiece" "protobuf"

print("Done. Runtime → Restart runtime now.")

In [ ]:
# ==============================================================================
# CELL 2.1b — Load ALLaM in 4-bit QLoRA
# ==============================================================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MODEL_ID  = "humain-ai/ALLaM-7B-Instruct-preview"
DRIVE_BASE = "/content/drive/MyDrive/allam_mcq_project"

# ── Verify GPU ────────────────────────────────────────────────────────────────
assert torch.cuda.is_available(), "No GPU found. Go to Runtime → Change runtime type → GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ── 4-bit quantization config ─────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# ── Load tokenizer ────────────────────────────────────────────────────────────
print("\nLoading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, padding_side="right")
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Vocab size : {tokenizer.vocab_size}")
print(f"Pad token  : {tokenizer.pad_token}")

# ── Load model — no patch needed with correct library versions ─────────────────
print("\nLoading ALLaM-7B in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False

print("\n" + "="*50)
print("Model loaded successfully.")
print(f"Dtype: {next(model.parameters()).dtype}")
print("="*50)

In [ ]:

# ==============================================================================
# CELL 2.1c — Attach LoRA Adapters
# ==============================================================================

from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)

print("LoRA adapters attached.")
model.print_trainable_parameters()
# Expected: ~0.5–2% trainable — if 0% something went wrong



In [ ]:
# ==============================================================================
# CELL 2.2 — Fine-Tuning
# ==============================================================================

from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

DRIVE_BASE  = "/content/drive/MyDrive/allam_mcq_project"
TRAIN_PATH  = f"{DRIVE_BASE}/data/train.jsonl"
VAL_PATH    = f"{DRIVE_BASE}/data/val.jsonl"
CKPT_DIR    = f"{DRIVE_BASE}/checkpoints"
ADAPTER_DIR = f"{DRIVE_BASE}/adapter"
MAX_SEQ_LEN = 1024

# ── Load datasets ─────────────────────────────────────────────────────────────
raw_train = load_dataset("json", data_files=TRAIN_PATH, split="train")
raw_val   = load_dataset("json", data_files=VAL_PATH,   split="train")

print(f"Train: {len(raw_train)} | Val: {len(raw_val)}")

# ── Convert messages to plain text strings BEFORE passing to trainer ──────────
# This avoids the collator seeing raw dicts entirely
def convert_to_text(example):
    text = ""
    for message in example["messages"]:
        text += f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>\n"
    return {"text": text}

train_dataset = raw_train.map(convert_to_text, remove_columns=raw_train.column_names)
val_dataset   = raw_val.map(convert_to_text,   remove_columns=raw_val.column_names)

print(f"Converted. Sample:\n{train_dataset[0]['text'][:200]}...")

# ── Sanity check ──────────────────────────────────────────────────────────────
assert "<|im_start|>system"     in train_dataset[0]["text"], "Missing system tag"
assert "<|im_start|>assistant"  in train_dataset[0]["text"], "Missing assistant tag"
print("Format check passed.")

# ── Training arguments ────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    eval_steps=50,
    save_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    dataloader_num_workers=0,
    remove_unused_columns=True,   # safe now — only "text" column remains
    optim="paged_adamw_8bit",
)

# ── Trainer — use dataset_text_field instead of formatting_func ───────────────
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",    # plain text column, no formatting_func needed
    args=training_args,
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

print(f"Starting training... checkpoints → {CKPT_DIR}")
trainer.train()

# ── Save adapter ──────────────────────────────────────────────────────────────
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f"\nAdapter saved → {ADAPTER_DIR}")
print("Training complete.")


In [ ]:
# ==============================================================================
# CELL 3.1 — Build FAISS RAG Index
# ==============================================================================

!pip install -q faiss-cpu sentence-transformers

from sentence_transformers import SentenceTransformer
import faiss, numpy as np, json, pickle

DRIVE_BASE = "/content/drive/MyDrive/allam_mcq_project"

with open(f"{DRIVE_BASE}/data/structured_rag_chunks.json", "r", encoding="utf-8") as f:
    rag_chunks = json.load(f)

print(f"Building index over {len(rag_chunks)} chunks...")

embedder   = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
texts      = [chunk["chunk_content"] for chunk in rag_chunks]
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=32)
embeddings = np.array(embeddings, dtype="float32")

dim   = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f"FAISS index: {index.ntotal} vectors, dim={dim}")

# Save to Drive
FAISS_PATH  = f"{DRIVE_BASE}/rag/rag_index.faiss"
CHUNKS_PATH = f"{DRIVE_BASE}/rag/rag_chunks.pkl"

faiss.write_index(index, FAISS_PATH)
with open(CHUNKS_PATH, "wb") as f:
    pickle.dump(rag_chunks, f)

print(f"Saved FAISS index → {FAISS_PATH}")
print(f"Saved chunks      → {CHUNKS_PATH}")



In [ ]:
# ==============================================================================
# X. Reload adapters in case of restart
# ==============================================================================
!pip install -q "transformers==4.41.2" "accelerate==0.34.0" "bitsandbytes" \
    "peft==0.11.0" "trl==0.9.4" "datasets==2.19.0" "sentencepiece" "protobuf" \
    "faiss-cpu" "sentence-transformers" "evaluate" "rouge-score" "gradio" "pypdf"

from google.colab import drive
drive.mount('/content/drive')

import os, torch, gc
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()

DRIVE_BASE  = "/content/drive/MyDrive/allam_mcq_project"
ADAPTER_DIR = f"{DRIVE_BASE}/adapter"
MODEL_ID    = "humain-ai/ALLaM-7B-Instruct-preview"
MAX_SEQ_LEN = 1024

free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9
print(f"Free VRAM: {free:.1f} GB")
print("Ready.")

In [ ]:
# ==============================================================================
# 3.2 RAG query + generation pipeline
# ==============================================================================

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import faiss, pickle, numpy as np, torch, json, gc
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# ── Load RAG components ───────────────────────────────────────────────────────
index = faiss.read_index(f"{DRIVE_BASE}/rag/rag_index.faiss")
with open(f"{DRIVE_BASE}/rag/rag_chunks.pkl", "rb") as f:
    rag_chunks = pickle.load(f)

# Force embedder to CPU
embedder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    device="cpu"
)
print(f"RAG index loaded: {index.ntotal} vectors")

# ── Load tokenizer ────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Clear before loading model ────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM before model load: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9:.1f} GB")

# ── Load base model in 4-bit ──────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    max_memory={0: "13GiB", "cpu": "8GiB"},
)

print("Attaching LoRA adapter...")
ft_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR,
    is_trainable=False,
)
ft_model.eval()

gc.collect()
torch.cuda.empty_cache()
print(f"Free VRAM after load: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9:.1f} GB")
print("Model ready.")

In [ ]:
# ==============================================================================
# Nahj MCQ Generator
# ==============================================================================

import os, gc
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

try:
    import torch
    torch.cuda.empty_cache()
    gc.collect()
except:
    pass

!pip install -q transformers==4.41.2 accelerate==0.34.0 bitsandbytes \
    peft==0.11.0 sentence-transformers faiss-cpu gradio pypdf

import gc, random, re, pickle, faiss, time
import numpy as np
import torch
import gradio as gr
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader
from collections import defaultdict
from huggingface_hub import hf_hub_download

gc.collect()
torch.cuda.empty_cache()
free = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9
print(f"Free VRAM at start: {free:.1f} GB")
assert free > 10, f"Not enough free VRAM ({free:.1f}GB). Runtime → Disconnect and delete runtime → reconnect."

# ── Config ────────────────────────────────────────────────────────────────────
ADAPTER_REPO = "jana-alaseeri/Nahj-mcq-pt1"
MODEL_ID     = "humain-ai/ALLaM-7B-Instruct-preview"
MAX_SEQ_LEN  = 768

# ── Download from HuggingFace ─────────────────────────────────────────────────
print("Downloading from HuggingFace...")
os.makedirs("/tmp/adapter", exist_ok=True)
for filename in [
    "adapter_model.safetensors", "adapter_config.json",
    "tokenizer.json", "tokenizer.model",
    "tokenizer_config.json", "special_tokens_map.json",
    "rag_index.faiss", "rag_chunks.pkl",
]:
    hf_hub_download(repo_id=ADAPTER_REPO, filename=filename, local_dir="/tmp/adapter")
ADAPTER_DIR = "/tmp/adapter"
print("Download complete.")

# ── Load RAG ──────────────────────────────────────────────────────────────────
index = faiss.read_index("/tmp/adapter/rag_index.faiss")
with open("/tmp/adapter/rag_chunks.pkl", "rb") as f:
    rag_chunks = pickle.load(f)
print(f"RAG index: {index.ntotal} vectors")

# ── Load embedder on CPU ──────────────────────────────────────────────────────
embedder = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device="cpu"
)

# ── Load tokenizer ────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# ── Load model ────────────────────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
print("Loading ALLaM-7B in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto",
    trust_remote_code=True, low_cpu_mem_usage=True,
    max_memory={0: "13GiB", "cpu": "8GiB"},
)
print("Attaching adapter...")
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
ft_model.eval()
gc.collect()
torch.cuda.empty_cache()
print("Model ready.")

# ==============================================================================
# HELPER FUNCTIONS
# ==============================================================================

def run_model(prompt: str, max_new_tokens: int = 150) -> str:
    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=MAX_SEQ_LEN
    ).to("cuda")
    with torch.no_grad():
        output = ft_model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, top_p=0.9, repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    raw = raw.split("<|im_end|>")[0].split("<|im_start|>")[0].strip()
    return raw


def clean_distractor(text: str, correct: str) -> str:
    """Aggressively strip all label prefixes and garbage from distractor lines."""
    if not text:
        return None
    text = re.sub(r'^(Wrong\s*[\d]?\s*[:\-]?\s*)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^(Incorrect\s*[\d]?\s*[:\-]?\s*)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^(Answer\s*[\d]?\s*[:\-]?\s*)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^[A-Da-d][\)\.:]\s*', '', text)
    text = re.sub(r'^[\d]+[\.\)]\s*', '', text)
    text = re.sub(r'^[\-\*\•]\s*', '', text)
    text = text.strip()
    # Reject bad values
    if not text:
        return None
    if text in ["—", "-", "N/A", "", "None", "null"]:
        return None
    if text.lower() == correct.lower():
        return None
    if len(text) < 4:
        return None
    # Reject if it still starts with "Wrong" or "Incorrect" after cleaning
    if re.match(r'^(Wrong|Incorrect|False)', text, re.IGNORECASE):
        return None
    return text


def parse_clo_input(clo_text):
    clos = {}
    if not clo_text or not clo_text.strip():
        return clos
    for line in clo_text.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        match = re.match(r'^(?:CLO\s*)?(\d+)\s*[:\-\.]\s*(.+)$', line, re.IGNORECASE)
        if match:
            clo_id, clo_desc = f"CLO{match.group(1)}", match.group(2).strip()
        else:
            clo_id, clo_desc = f"CLO{len(clos)+1}", line
        clos[clo_id] = clo_desc
    return clos


def map_chunks_to_clos(chunks, user_clos):
    if not user_clos:
        return [(chunk, "General", "General understanding") for chunk in chunks]
    clo_ids    = list(user_clos.keys())
    clo_descs  = list(user_clos.values())
    clo_embs   = embedder.encode(clo_descs,  convert_to_numpy=True).astype("float32")
    chunk_embs = embedder.encode(chunks,     convert_to_numpy=True).astype("float32")
    dim        = clo_embs.shape[1]
    tmp_index  = faiss.IndexFlatL2(dim)
    tmp_index.add(clo_embs)
    _, indices = tmp_index.search(chunk_embs, 1)
    return [
        (chunk, clo_ids[indices[i][0]], clo_descs[indices[i][0]])
        for i, chunk in enumerate(chunks)
    ]


def parse_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    chunks, current, word_count = [], [], 0
    for page in reader.pages:
        text  = page.extract_text() or ""
        words = text.split()
        for word in words:
            current.append(word)
            word_count += 1
            if word_count >= 250:
                chunks.append(" ".join(current))
                current, word_count = [], 0
    if current:
        chunks.append(" ".join(current))
    return [c for c in chunks if len(c.strip()) > 50]


def generate_mcq(context_chunk: str, clo_desc: str = None):
    """
    Step 1: question + correct answer.
    Step 2: 3 distractors.
    Returns (question, correct_answer, [d1, d2, d3])
    """
    clo_line = ""
    if clo_desc and clo_desc.lower() not in ["general understanding", "", "none"]:
        clo_line = f"Focus on: {clo_desc}.\n"

    # Step 1 ───────────────────────────────────────────────────────────────────
    prompt1 = (
        f"<|im_start|>system\n"
        f"You are an educational assistant. Generate one multiple choice question "
        f"based only on the content provided.<|im_end|>\n"
        f"<|im_start|>user\n"
        f"{clo_line}"
        f"Content:\n{context_chunk}\n\n"
        f"Generate one multiple-choice question with options A, B, C, D "
        f"and indicate the correct answer.<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    raw1 = run_model(prompt1, max_new_tokens=150)

    q_match  = re.search(r'Question:\s*(.+?)(?=\nA[\)\.:]|\Z)', raw1, re.DOTALL)
    question = q_match.group(1).strip() if q_match else raw1.strip()
    question = question.split("\n")[0].strip()

    correct_answer = None
    a_match = re.search(r'A[\)\.:]\s*(.+?)(?=\nB[\)\.:]|\nCorrect|\Z)', raw1, re.DOTALL)
    if a_match:
        correct_answer = a_match.group(1).strip().split("\n")[0].strip()

    if not correct_answer:
        c_line = re.search(r'Correct(?:\s*Answer)?:\s*([ABCD])', raw1)
        if c_line:
            letter = c_line.group(1)
            lm = re.search(
                rf'{letter}[\)\.:]\s*(.+?)(?=\n[ABCD][\)\.]|\nCorrect|\Z)',
                raw1, re.DOTALL
            )
            if lm:
                correct_answer = lm.group(1).strip().split("\n")[0].strip()

    if not question or not correct_answer:
        return None, None, []

    # Step 2: distractors ──────────────────────────────────────────────────────
    prompt2 = (
        f"<|im_start|>system\nYou are an educational assistant.<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Content: {context_chunk[:300]}\n"
        f"Question: {question}\n"
        f"Correct answer: {correct_answer}\n\n"
        f"Write 3 short incorrect answers. "
        f"Write only the answer text, one per line, no numbering, no labels:\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    raw2 = run_model(prompt2, max_new_tokens=80)

    distractors = []
    for line in raw2.split("\n"):
        cleaned = clean_distractor(line.strip(), correct_answer)
        if cleaned and cleaned not in distractors:
            distractors.append(cleaned)
        if len(distractors) == 3:
            break

    fallbacks = [
        "None of the above",
        "All of the above",
        "Cannot be determined from the content",
    ]
    for fb in fallbacks:
        if len(distractors) >= 3:
            break
        if fb not in distractors:
            distractors.append(fb)

    return question, correct_answer, distractors[:3]


def build_mcq_dict(question, correct_answer, distractors, clo_id, clo_desc):
    all_options    = [correct_answer] + distractors
    random.shuffle(all_options)
    options_dict   = {}
    correct_letter = None
    for i, letter in enumerate(["A", "B", "C", "D"]):
        options_dict[letter] = all_options[i]
        if all_options[i] == correct_answer:
            correct_letter = letter
    return {
        "question": question, "options": options_dict,
        "correct": correct_letter, "clo_id": clo_id, "clo_desc": clo_desc,
    }


def analyze_weaknesses(session_results, user_clos):
    if not session_results:
        return "No results to analyze yet."
    clo_stats = defaultdict(lambda: {"correct": 0, "wrong": 0, "desc": ""})
    for r in session_results:
        clo_id   = r["clo_id"]
        is_right = r["user_answer"].upper() == r["correct_answer"].upper()
        clo_stats[clo_id]["desc"] = r.get("clo_desc", "")
        if is_right:
            clo_stats[clo_id]["correct"] += 1
        else:
            clo_stats[clo_id]["wrong"] += 1
    sorted_clos  = sorted(clo_stats.items(), key=lambda x: x[1]["wrong"], reverse=True)
    report_lines = ["## 📊 CLO Performance Report\n"]
    for clo_id, stats in sorted_clos:
        total    = stats["correct"] + stats["wrong"]
        score    = stats["correct"] / total * 100 if total > 0 else 0
        emoji    = "✅" if score >= 70 else "⚠️" if score >= 40 else "❌"
        bar      = "█" * int(score/10) + "░" * (10 - int(score/10))
        desc     = stats["desc"] or user_clos.get(clo_id, "")
        report_lines.append(
            f"{emoji} **{clo_id}**: _{desc[:100]}_\n"
            f"   Score: {score:.0f}% [{bar}] ({stats['correct']}/{total})\n"
        )
    weak_clos = [
        (cid, s) for cid, s in sorted_clos
        if (s["correct"] / (s["correct"] + s["wrong"])) < 0.5
    ]
    if weak_clos:
        report_lines.append("\n### 🎯 Focus Areas (below 50%)")
        for clo_id, stats in weak_clos:
            report_lines.append(f"- **{clo_id}**: {stats['desc'][:100]}")
        report_lines.append("\n💡 *Click Practice Weak CLOs for targeted questions.*")
    else:
        report_lines.append("\n### 🎉 All CLOs above 50% — great work!")
    return "\n".join(report_lines)


# ── Session state ─────────────────────────────────────────────────────────────
session_state = {
    "questions":     [],
    "current_index": 0,
    "results":       [],
    "user_clos":     {},
}


def generate_from_pdf(pdf_file, num_questions, clo_text, progress=gr.Progress()):
    # FIXED: always cast to int
    num_questions = max(1, int(num_questions or 5))
    print(f"Generating {num_questions} questions...")

    if pdf_file is None:
        return (
            "Please upload a PDF file.", "",
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)
        )

    session_state.update({"questions": [], "current_index": 0, "results": []})
    user_clos = parse_clo_input(clo_text)
    session_state["user_clos"] = user_clos

    chunks = parse_pdf(pdf_file.name)
    if not chunks:
        return (
            "Could not extract text from this PDF.", "",
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)
        )

    num_questions = min(num_questions, len(chunks))
    selected      = random.sample(chunks, num_questions)
    mapped        = map_chunks_to_clos(selected, user_clos)

    # Progress bar
    progress(0, desc="Starting generation...")

    for i, (chunk, clo_id, clo_desc) in enumerate(mapped):
        pct = i / num_questions
        progress(pct, desc=f"Generating question {i+1} of {num_questions}...")
        gr.Info(f"⏳ Question {i+1} of {num_questions} — {clo_id}")
        print(f"  Q{i+1}/{num_questions} → {clo_id}")

        t0 = time.time()
        question, correct_answer, distractors = generate_mcq(chunk, clo_desc=clo_desc)
        elapsed = time.time() - t0
        print(f"  ✓ done in {elapsed:.1f}s")

        if question and correct_answer and len(distractors) == 3:
            mcq = build_mcq_dict(question, correct_answer, distractors, clo_id, clo_desc)
        else:
            mcq = {
                "question": question or "Could not generate question.",
                "options":  {
                    "A": correct_answer or "N/A",
                    "B": "None of the above",
                    "C": "All of the above",
                    "D": "Cannot be determined",
                },
                "correct": "A", "clo_id": clo_id, "clo_desc": clo_desc,
            }
        session_state["questions"].append(mcq)

    progress(1.0, desc="Done!")
    gr.Info(f"✅ All {num_questions} questions ready!")
    print("Generation complete.")
    return show_current_question()


def show_current_question():
    questions = session_state["questions"]
    idx       = session_state["current_index"]

    if not questions:
        return (
            "No questions generated.", "",
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)
        )

    if idx >= len(questions):
        report = analyze_weaknesses(session_state["results"], session_state["user_clos"])
        return (
            "## ✅ Quiz Complete!\n\n" + report,
            "",
            gr.update(visible=False),       # hide answer radio
            gr.update(visible=False),       # hide submit button
            gr.update(visible=True),        # show practice button
        )

    q    = questions[idx]
    text = (
        f"### Question {idx+1} of {len(questions)}\n"
        f"**{q.get('clo_id','General')}**: _{q.get('clo_desc','')[:80]}_\n\n"
        f"---\n\n"
        f"**{q['question']}**\n\n"
        f"**A)** {q['options']['A']}\n\n"
        f"**B)** {q['options']['B']}\n\n"
        f"**C)** {q['options']['C']}\n\n"
        f"**D)** {q['options']['D']}\n"
    )
    return (
        text,
        "",
        gr.update(visible=True, value=None),
        gr.update(visible=True),
        gr.update(visible=False),
    )


def submit_answer(user_answer):
    questions = session_state["questions"]
    idx       = session_state["current_index"]

    if not questions or idx >= len(questions):
        return (
            "No active question.", "",
            gr.update(visible=False), gr.update(visible=False), gr.update(visible=False)
        )

    if not user_answer:
        return (
            show_current_question()[0],
            "⚠️ Please select an answer (A, B, C, or D) first.",
            gr.update(visible=True), gr.update(visible=True), gr.update(visible=False)
        )

    # Get current question BEFORE advancing index
    q       = questions[idx]
    correct = q["correct"].upper()
    chosen  = user_answer.upper()

    # Record result
    session_state["results"].append({
        "clo_id":         q["clo_id"],
        "clo_desc":       q.get("clo_desc", ""),
        "user_answer":    chosen,
        "correct_answer": correct,
    })

    # Build feedback from CURRENT question options
    if chosen == correct:
        feedback = f"✅ Correct! The answer is **{correct}) {q['options'][correct]}**"
    else:
        feedback = (
            f"❌ Incorrect. You chose **{chosen}) {q['options'].get(chosen, '?')}**. "
            f"The correct answer is **{correct}) {q['options'][correct]}**"
        )

    # Advance index AFTER building feedback
    session_state["current_index"] += 1

    # Get next question state
    next_q, _, radio_vis, submit_vis, practice_vis = show_current_question()
    return next_q, feedback, radio_vis, submit_vis, practice_vis


def practice_weak_clos():
    results   = session_state["results"]
    user_clos = session_state["user_clos"]

    if not results:
        return (
            "No session data yet. Complete a quiz first.",
            "",
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
        )

    clo_stats = defaultdict(lambda: {"correct": 0, "wrong": 0})
    for r in results:
        if r["user_answer"].upper() == r["correct_answer"].upper():
            clo_stats[r["clo_id"]]["correct"] += 1
        else:
            clo_stats[r["clo_id"]]["wrong"] += 1

    weak_clo_ids = [
        cid for cid, s in clo_stats.items()
        if s["wrong"] > s["correct"]
    ]
    if not weak_clo_ids:
        return (
            "No weak CLOs detected — you're doing great! 🎉",
            "",
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(visible=True),
        )

    session_state.update({"questions": [], "current_index": 0, "results": []})

    weak_chunks = []
    for clo_id in weak_clo_ids:
        clo_desc = user_clos.get(clo_id, clo_id)
        q_emb    = embedder.encode([clo_desc], convert_to_numpy=True).astype("float32")
        _, idxs  = index.search(q_emb, 5)
        for i in idxs[0]:
            weak_chunks.append({
                "content":  rag_chunks[i]["chunk_content"],
                "clo_id":   clo_id,
                "clo_desc": clo_desc,
            })

    selected = random.sample(weak_chunks, min(5, len(weak_chunks)))
    print(f"Generating practice questions for weak CLOs: {weak_clo_ids}")

    for i, item in enumerate(selected):
        gr.Info(f"⏳ Practice question {i+1} of {len(selected)}...")
        question, correct_answer, distractors = generate_mcq(
            item["content"], clo_desc=item["clo_desc"]
        )
        if question and correct_answer and len(distractors) == 3:
            mcq = build_mcq_dict(
                question, correct_answer, distractors,
                item["clo_id"], item["clo_desc"]
            )
            session_state["questions"].append(mcq)

    q_text, _, _, _, _ = show_current_question()

    header = (
        "## 🎯 Targeted Practice\n"
        + "\n".join([f"- **{c}**: {user_clos.get(c,'')}" for c in weak_clo_ids])
        + "\n\n"
        + q_text
    )
    return (
        header,
        "",
        gr.update(visible=True, value=None),
        gr.update(visible=True),
        gr.update(visible=False),
    )


# ==============================================================================
# GRADIO UI
# ==============================================================================
with gr.Blocks(title="Nahj: MCQ Generator", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 📚 Nahj: MCQ Generator
    ### Powered by ALLaM-7B (Fine-tuned) + RAG + CLO Assessment
    Upload any educational PDF, define your CLOs, and get a personalized quiz.
    """)

    with gr.Row():
        # ── LEFT: controls ────────────────────────────────────────────────────
        with gr.Column(scale=1):
            pdf_input = gr.File(label="📄 Upload PDF", file_types=[".pdf"])
            gr.Markdown("### 🎯 Define Your CLOs *(optional)*")
            gr.Markdown("*One per line: `CLO1: description` or just the description*")
            clo_input = gr.Textbox(
                label="Course Learning Outcomes",
                placeholder=(
                    "CLO1: Understand basic data structures\n"
                    "CLO2: Apply sorting algorithms\n"
                    "CLO3: Analyze time complexity"
                ),
                lines=5,
            )
            num_q = gr.Slider(
                minimum=1, maximum=20, value=5,
                step=1, label="Number of Questions",
                interactive=True,
            )
            generate_btn = gr.Button("🚀 Generate Questions", variant="primary", size="lg")
            gr.Markdown("---")
            practice_btn = gr.Button("🎯 Practice Weak CLOs", variant="stop", visible=False)

        # ── RIGHT: question + answer + feedback ───────────────────────────────
        with gr.Column(scale=2):
            question_display = gr.Markdown(
                "*Upload a PDF, optionally define CLOs, and click Generate.*"
            )
            answer_input = gr.Radio(
                choices=["A", "B", "C", "D"],
                label="✍️ Your Answer",
                visible=False,
                interactive=True,
            )
            submit_btn = gr.Button(
                "Submit Answer ✅",
                variant="primary",
                visible=False,
            )
            feedback_box = gr.Markdown("")

    # ── Wiring — all functions return 5 outputs ───────────────────────────────
    generate_btn.click(
        fn=generate_from_pdf,
        inputs=[pdf_input, num_q, clo_input],
        outputs=[question_display, feedback_box, answer_input, submit_btn, practice_btn],
    )
    submit_btn.click(
        fn=submit_answer,
        inputs=[answer_input],
        outputs=[question_display, feedback_box, answer_input, submit_btn, practice_btn],
    )
    practice_btn.click(
        fn=practice_weak_clos,
        inputs=[],
        outputs=[question_display, feedback_box, answer_input, submit_btn, practice_btn],
    )

    gr.Markdown("""
    ---
    **How it works:**
    Upload PDF → Define CLOs → App maps chunks to CLOs → Quiz →
    CLO Performance Report → Practice Weak CLOs
    """)

demo.launch(share=True, debug=True)